# Nova Workshop 2026
<br/>
<img src="https://www.polyestertime.com/wp-content/uploads/2017/01/Nova-Chemical-23-09-2016.jpg" />
<br/><br/>

## Introduction
Now that we have everything up and running, we've accomplished the following:
- Tested our integration with Genie Code
- Created a synthetic dataset
- Built a basic lakehouse foundation
- Configured a Lakebase database, schema and table for use with an interactive application

All we need to do now is run a few remianing configuration items and validate our data and connectivity.

### Goal:
- Implement helper functions that the Databricks App will reuse:
  - Load KPIs for a `line_id`.
  - Load incidents for a `line_id`.
  - Insert a new incident.

In [0]:
import pyspark.sql.functions as F

user_schema = "andrij_demo"      # TODO
catalog_name = "nova_workshop"
lakebase_db = "nova-incidents"

gold_table = f"{catalog_name}.{user_schema}.gold_daily_line_kpis"
incidents_table = f"{catalog_name}.{user_schema}.gold_incidents"

print("Gold KPIs table :", gold_table)
print("Incidents table :", incidents_table)

## Helper: load_kpis(line_id)

This cell defines a reusable helper function to load KPIs for a given `line_id`. By encapsulating this logic in a function, we make our code more modular, easier to maintain, and simpler to test. This approach also enables us to quickly adapt or extend KPI loading logic in the future, and makes it straightforward to reuse this functionality across multiple notebooks or applications.

In [0]:
# The load_kpis function retrieves daily KPI records for a specific production line from the gold KPIs table.
# It reads the table as a Spark DataFrame, filters rows where 'line_id' matches the provided argument,
# and orders the results by the 'day' column. This allows for easy extraction and analysis of KPIs for any line.
# Using a helper function like this improves code reusability, readability, and maintainability,
# enabling consistent data access patterns and simplifying future modifications.

def load_kpis(line_id: str):
    return (
        spark.read.table(gold_table)
            .filter(F.col("line_id") == line_id)
            .orderBy("day")
    )

def load_incidents(line_id: str):
    return (
        spark.read.table(incidents_table)
            .filter(F.col("line_id") == line_id)
            .orderBy("ts")
    )

# Test
display(load_kpis("LINE_01").limit(10))

### Why Use Abstract Classes and Concrete Implementations?

Opting for an abstract base class with concrete implementations is a best-practice design pattern that offers several advantages:

- **Extensibility**: Abstract classes define a common interface and shared behavior, allowing new functionality to be added by creating new subclasses without modifying existing code. This makes it easy to extend the system as requirements evolve.

- **Reusability**: By encapsulating shared logic in the abstract base class, code can be reused across multiple concrete implementations. This reduces duplication and promotes consistency.

- **Durability**: Abstract classes enforce structure and contracts, making code more robust and less prone to errors. Changes to the base class propagate safely to all subclasses, improving maintainability and reducing the risk of breaking existing functionality.

#### Further Reading
- [Abstract Classes vs Interfaces in Python](https://realpython.com/python-interface/)
- [Object-Oriented Programming: Abstract Classes](https://docs.python.org/3/library/abc.html)
- [Design Patterns: Template Method](https://refactoring.guru/design-patterns/template-method)

In [0]:
from abc import ABC, abstractmethod

# Abstract base class defining the KPI loader interface
# The @abstractmethod decorator marks methods that must be implemented by subclasses.
# It enforces that any subclass of BaseKpiLoader must provide its own implementation of load_kpis.
class BaseKpiLoader(ABC):
    @abstractmethod
    def load_kpis(self, line_id: str):
        # Abstract method to load KPIs for a given line_id
        pass

# Concrete implementation of BaseKpiLoader for loading KPIs from a Spark table
class KpiLoader(BaseKpiLoader):
    def __init__(self, table_name):
        # Store the table name to be used for loading KPIs
        self.table_name = table_name

    def load_kpis(self, line_id: str):
        # Load KPIs for the specified line_id from the Spark table
        return (
            spark.read.table(self.table_name)
                .filter(F.col("line_id") == line_id)
                .orderBy("day")
        )

# Test
kpi_loader = KpiLoader(gold_table)
display(kpi_loader.load_kpis("LINE_01").limit(10))

## Helper: insert_incident(...)

Use Genie here to generate safe insert logic for Lakebase. Feel free to use either the helper function or challenge yourself by extending the functionality of the abstract base class.

### Suggested prompt:

> Write a Python function `insert_incident(line_id, severity, summary, details)` that inserts a new row into `${incidents_table}` with `ts = current_timestamp()` and `status = 'OPEN'`.
> Use Spark SQL or JDBC as appropriate.


In [0]:
# PLACEHOLDER: implement via Genie or manually.

# def insert_incident(line_id: str, severity: str, summary: str, details: str):
#     # Example using Spark SQL:
#     spark.sql(f"""
#       INSERT INTO {incidents_table} (line_id, ts, severity, summary, details, status)
#       VALUES ('{line_id}', current_timestamp(), '{severity}', '{summary}', '{details}', 'OPEN')
#     """)
#     return True

# # Quick smoke test (consider commenting out to avoid spam)
# insert_incident("LINE_01", "MEDIUM", "Test incident", "This is a test.")

display(load_incidents("LINE_01").limit(500))


# Summary

In this notebook, we accomplished the following key steps to establish a robust foundation for our Databricks application with Lakebase:

- **Tested Genie Code Integration**: Verified that our environment is correctly set up to interact with Genie, ensuring seamless code generation and automation.
- **Created a Synthetic Dataset**: Built a sample dataset to simulate real-world data scenarios, enabling us to test and validate our workflows.
- **Lakehouse Foundation**: Established the basic lakehouse architecture, providing a scalable and flexible platform for data storage and analytics.
- **Configured Lakebase Database, Schema, and Table**: Set up the necessary Lakebase structures to support our application's data requirements.
- **Developed Helper Functions**: Implemented reusable functions for loading KPIs, retrieving incidents, and inserting new incidents, promoting modularity and maintainability.
- **Explored Abstract Classes and Concrete Implementations**: Discussed best practices for designing extensible and durable code using object-oriented principles.

---

This comprehensive back-end connectivity test confirms that our environment and data pipelines are fully operational. With these foundational elements in place, we are well-prepared to continue building and scaling our Databricks application with Lakebase.